In [ ]:
#  Imports 
import gc, json, math, os, random, shutil
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision.utils import make_grid, save_image
from tqdm import tqdm




In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch : {torch.__version__}")
print(f"Device  : {device}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")

# Paths 
DATA_DIR   = Path("/kaggle/input/datasets/rishu2204/gravitational-lensing/Samples")
OUTPUT_DIR = Path("/kaggle/working/vae_outputs")
CKPT_DIR   = Path("/kaggle/working")          # root 
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:

#  Hyperparameters 
IMAGE_SIZE   = 64
LATENT_DIM   = 128
BATCH_SIZE   = 32
EPOCHS       = 60
LR           = 3e-4
BETA_KL      = 0.001       # KL weight (low → sharper images)
NUM_WORKERS  = 2
SAVE_EVERY   = 10          # checkpoint every N epochs

In [ ]:
#  Dataset
class LensingDataset(Dataset):
    def __init__(self, root, image_size=64):
        self.files = sorted(Path(root).rglob("*.npy"))
        self.sz    = image_size
        if not self.files:
            raise FileNotFoundError(f"No .npy files under {root}")
        print(f"Found {len(self.files)} files")

    def __len__(self): return len(self.files)

    def __getitem__(self, idx):
        arr = np.load(self.files[idx]).astype(np.float32)
        if arr.ndim == 2:           arr = arr[np.newaxis]
        elif arr.shape[-1] == 1:    arr = arr.transpose(2, 0, 1)
        img = torch.from_numpy(arr)
        if img.shape[-1] != self.sz:
            img = F.interpolate(img.unsqueeze(0), self.sz,
                                mode="bilinear", align_corners=False).squeeze(0)
        lo, hi = img.min(), img.max()
        img = 2*(img-lo)/(hi-lo+1e-6) - 1 if (hi-lo) > 1e-6 else torch.zeros_like(img)
        return img

ds        = LensingDataset(DATA_DIR, IMAGE_SIZE)
val_n     = max(1, len(ds)//10)
train_ds, val_ds = random_split(ds, [len(ds)-val_n, val_n],
                                generator=torch.Generator().manual_seed(42))
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=False, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=False)
print(f"Train: {len(train_ds)}  Val: {len(val_ds)}")

In [ ]:
# Model 
class ResBlock(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.GroupNorm(min(8, ch), ch), nn.SiLU(),
            nn.Conv2d(ch, ch, 3, padding=1),
            nn.GroupNorm(min(8, ch), ch), nn.SiLU(),
            nn.Conv2d(ch, ch, 3, padding=1),
        )
    def forward(self, x): return x + self.net(x)


class Encoder(nn.Module):
    def __init__(self, in_ch=1, base=64, latent_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, base, 3, padding=1),       # 64×64
            ResBlock(base),
            nn.Conv2d(base, base*2, 4, 2, 1),            # 32×32
            ResBlock(base*2),
            nn.Conv2d(base*2, base*4, 4, 2, 1),          # 16×16
            ResBlock(base*4),
            nn.Conv2d(base*4, base*4, 4, 2, 1),          # 8×8
            ResBlock(base*4),
            nn.GroupNorm(min(32, base*4), base*4), nn.SiLU(),
            nn.Flatten(),
        )
        flat = base*4 * 8 * 8
        self.mu      = nn.Linear(flat, latent_dim)
        self.log_var = nn.Linear(flat, latent_dim)

    def forward(self, x):
        h = self.net(x)
        return self.mu(h), self.log_var(h)




In [ ]:
class Decoder(nn.Module):
    def __init__(self, out_ch=1, base=64, latent_dim=128):
        super().__init__()
        flat = base*4 * 8 * 8
        self.proj = nn.Linear(latent_dim, flat)
        self.base = base
        self.net  = nn.Sequential(
            ResBlock(base*4),
            nn.ConvTranspose2d(base*4, base*4, 4, 2, 1),  # 16×16
            ResBlock(base*4),
            nn.ConvTranspose2d(base*4, base*2, 4, 2, 1),  # 32×32
            ResBlock(base*2),
            nn.ConvTranspose2d(base*2, base,   4, 2, 1),  # 64×64
            ResBlock(base),
            nn.GroupNorm(min(8, base), base), nn.SiLU(),
            nn.Conv2d(base, out_ch, 3, padding=1),
            nn.Tanh(),
        )

    def forward(self, z):
        h = self.proj(z).view(z.size(0), self.base*4, 8, 8)
        return self.net(h)

In [ ]:
class VAE(nn.Module):
    def __init__(self, latent_dim=128, base=64):
        super().__init__()
        self.encoder = Encoder(latent_dim=latent_dim, base=base)
        self.decoder = Decoder(latent_dim=latent_dim, base=base)

    def reparameterise(self, mu, log_var):
        std = (0.5 * log_var).exp()
        return mu + std * torch.randn_like(std)

    def forward(self, x):
        mu, log_var = self.encoder(x)
        z    = self.reparameterise(mu, log_var)
        recon = self.decoder(z)
        return recon, mu, log_var

    def sample(self, n, device):
        z = torch.randn(n, self.encoder.mu.out_features, device=device)
        return self.decoder(z)

In [ ]:
model = VAE(latent_dim=LATENT_DIM, base=64).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"VAE parameters: {n_params:,}")

In [ ]:
# Loss 
def vae_loss(recon, x, mu, log_var, beta=BETA_KL):
    recon_loss = F.l1_loss(recon, x)           # L1 → sharper than MSE
    kl         = -0.5 * (1 + log_var - mu**2 - log_var.exp()).mean()
    return recon_loss + beta * kl, recon_loss.item(), kl.item()

In [ ]:
#  Training 
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)
scaler    = torch.amp.GradScaler(enabled=device.type=="cuda")

train_losses, val_losses = [], []
best_val = float("inf")

print("\nStarting training …")
for epoch in range(1, EPOCHS+1):
    # train 
    model.train()
    ep_loss = 0.0
    for x in tqdm(train_loader, desc=f"Ep {epoch}/{EPOCHS}", leave=False):
        x = x.to(device)
        with torch.amp.autocast(device_type=device.type, enabled=device.type=="cuda"):
            recon, mu, log_var = model(x)
            loss, _, _ = vae_loss(recon, x, mu, log_var)
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer); scaler.update()
        ep_loss += loss.item()
        del x, recon, loss
    scheduler.step()
    train_losses.append(ep_loss / len(train_loader))

    #  val 
    model.eval(); vl = 0.0
    with torch.no_grad():
        for x in val_loader:
            x = x.to(device)
            with torch.amp.autocast(device_type=device.type, enabled=device.type=="cuda"):
                recon, mu, log_var = model(x)
                l, _, _ = vae_loss(recon, x, mu, log_var)
            vl += l.item(); del x, recon, l
    val_losses.append(vl / len(val_loader))

    print(f"Epoch {epoch:3d}  train={train_losses[-1]:.4f}  val={val_losses[-1]:.4f}")

    # checkpoint 
    ck = {"epoch": epoch, "model": model.state_dict(),
          "opt": optimizer.state_dict(), "val_loss": val_losses[-1]}

    if epoch % SAVE_EVERY == 0 or epoch == EPOCHS:
        torch.save(ck, CKPT_DIR / "vae_latest.pt")
        print(f"  ✓ Checkpoint saved → /kaggle/working/vae_latest.pt")

    if val_losses[-1] < best_val:
        best_val = val_losses[-1]
        torch.save(ck, CKPT_DIR / "vae_best.pt")
        print(f"   Best model saved → /kaggle/working/vae_best.pt  (val={best_val:.4f})")

    torch.cuda.empty_cache(); gc.collect()

print(f"\n Best val loss: {best_val:.4f}")

In [ ]:
# Generate & save all outputs 
print("\nLoading best checkpoint …")
ck    = torch.load(CKPT_DIR / "vae_best.pt", map_location=device, weights_only=False)
model.load_state_dict(ck["model"])
model.eval()

NUM_GEN = 256
print(f"Generating {NUM_GEN} images …")
fake_batches = []
with torch.no_grad():
    for _ in range(0, NUM_GEN, 32):
        n = min(32, NUM_GEN - sum(x.shape[0] for x in fake_batches))
        fake_batches.append(model.sample(n, device).cpu())
fake_samples = torch.cat(fake_batches)[:NUM_GEN]



In [ ]:
# Real samples
real_batches = []
for x in val_loader:
    real_batches.append(x)
    if sum(b.shape[0] for b in real_batches) >= NUM_GEN: break
real_samples = torch.cat(real_batches)[:NUM_GEN]

print(f"Fake: {fake_samples.shape}  Real: {real_samples.shape}")



In [ ]:
# Save grids
def save_grid(t, path, nrow=8):
    grid = make_grid(((t.clamp(-1,1)+1)/2), nrow=nrow, padding=2)
    save_image(grid, path)

save_grid(fake_samples[:64], OUTPUT_DIR/"generated_grid.png")
save_grid(real_samples[:64], OUTPUT_DIR/"real_grid.png")
print(" saved generated_grid.png and real_grid.png")

# Save 64 individual generated PNGs
ind = OUTPUT_DIR/"generated_individual"; ind.mkdir(exist_ok=True)
for i, img in enumerate(fake_samples[:64]):
    save_image(((img.clamp(-1,1)+1)/2), ind/f"{i:04d}.png")
print(f" saved 64 individual images → {ind}")

In [ ]:
# CELL 7: Metrics 
def radial_profile(img):
    img = img.float()
    h, w = img.shape
    y, x = torch.meshgrid(torch.arange(h), torch.arange(w), indexing="ij")
    r = ((x-(w-1)/2)**2 + (y-(h-1)/2)**2).sqrt().long()
    mx = int(r.max())+1
    p = torch.zeros(mx); c = torch.zeros(mx)
    p.scatter_add_(0, r.flatten(), img.flatten())
    c.scatter_add_(0, r.flatten(), torch.ones(img.numel()))
    return p / c.clamp(min=1)



In [ ]:
def rp_rmse(real, fake):
    pr = torch.stack([radial_profile(x[0]) for x in real]).mean(0)
    pf = torch.stack([radial_profile(x[0]) for x in fake]).mean(0)
    n  = min(len(pr), len(pf))
    return float(((pr[:n]-pf[:n])**2).mean().sqrt())

def ps_rmse(real, fake):
    def ps(imgs):
        return torch.stack([
            radial_profile(torch.fft.fftshift(torch.fft.fft2(x[0].float())).abs()**2)
            for x in imgs]).mean(0)
    pr = np.log1p(ps(real).numpy())
    pf = np.log1p(ps(fake).numpy())
    n  = min(len(pr), len(pf))
    return float(np.sqrt(np.mean((pr[:n]-pf[:n])**2)))



In [ ]:
N = min(128, NUM_GEN)
print("Computing physics metrics …")
rp = rp_rmse(real_samples[:N], fake_samples[:N])
ps = ps_rmse(real_samples[:N], fake_samples[:N])
print(f"  Radial Profile RMSE : {rp:.6f}")
print(f"  Power Spectrum RMSE : {ps:.6f}")

In [ ]:
# CELL 8: FID
from torchvision.models import inception_v3, Inception_V3_Weights
from scipy import linalg as la


inc = inception_v3(weights=Inception_V3_Weights.IMAGENET1K_V1)
inc.fc = nn.Identity(); inc.aux_logits = False
inc = inc.to(device).eval()

@torch.no_grad()
def get_feats(imgs, batch=64):
    feats = []
    for i in range(0, len(imgs), batch):
        b = imgs[i:i+batch].to(device)
        if b.shape[1] == 1: b = b.repeat(1,3,1,1)
        b = F.interpolate(b, (299,299), mode="bilinear", align_corners=False)
        b = ((b.clamp(-1,1)+1)/2)
        feats.append(inc(b).cpu().numpy())
    return np.concatenate(feats, 0).astype(np.float32)



In [ ]:
def fid(rf, ff):
    m1,s1 = rf.mean(0), np.cov(rf, rowvar=False)
    m2,s2 = ff.mean(0), np.cov(ff, rowvar=False)
    diff  = m1-m2
    cm,_  = la.sqrtm(s1@s2, disp=False)
    if np.iscomplexobj(cm): cm = cm.real
    return float(diff@diff + np.trace(s1+s2-2*cm))

rf = get_feats(real_samples); ff = get_feats(fake_samples)
fid_score = fid(rf, ff)
del inc, rf, ff; torch.cuda.empty_cache(); gc.collect()
print(f"  FID : {fid_score:.2f}")

In [ ]:
# Loss curve
plt.figure(figsize=(10,4))
plt.plot(train_losses, label="Train", lw=2)
plt.plot(val_losses,   label="Val",   lw=2)
plt.xlabel("Epoch"); plt.ylabel("Loss")
plt.title("VAE Training Loss"); plt.legend(); plt.tight_layout()
plt.savefig(OUTPUT_DIR/"loss_curve.png", dpi=120); plt.close()
print(" saved loss_curve.png")



In [ ]:
# Comparison grid
fig, axes = plt.subplots(4, 8, figsize=(20,10))
axes = axes.flatten()
for i in range(16):
    axes[i].imshow(((real_samples[i,0]+1)/2).clamp(0,1).numpy(), cmap="inferno", origin="lower")
    axes[i].axis("off")
    if i==0: axes[i].set_title("Real", fontsize=10, color="white")
for i in range(16):
    axes[16+i].imshow(((fake_samples[i,0]+1)/2).clamp(0,1).numpy(), cmap="inferno", origin="lower")
    axes[16+i].axis("off")
    if i==0: axes[16+i].set_title("Generated (VAE)", fontsize=10, color="white")
plt.suptitle("Strong Gravitational Lensing — Real vs VAE Generated", fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUT_DIR/"comparison.png", dpi=150, bbox_inches="tight", facecolor="black")
plt.close(); print(" saved comparison.png")



In [ ]:
# Radial profile
rp_r = torch.stack([radial_profile(x[0]) for x in real_samples[:64]]).mean(0).numpy()
rp_f = torch.stack([radial_profile(x[0]) for x in fake_samples[:64]]).mean(0).numpy()
n = min(len(rp_r), len(rp_f))
plt.figure(figsize=(8,4))
plt.plot(rp_r[:n], label="Real",      lw=2)
plt.plot(rp_f[:n], label="Generated", lw=2, ls="--")
plt.fill_between(range(n), rp_r[:n], rp_f[:n], alpha=0.2)
plt.xlabel("Radial distance (px)"); plt.ylabel("Mean flux")
plt.title("Azimuthally-averaged radial profile"); plt.legend(); plt.tight_layout()
plt.savefig(OUTPUT_DIR/"radial_profiles.png", dpi=120); plt.close()
print(" saved radial_profiles.png")



In [ ]:
# Pixel dist
rp2 = ((real_samples+1)/2).clamp(0,1).flatten().numpy()
fp2 = ((fake_samples+1)/2).clamp(0,1).flatten().numpy()
plt.figure(figsize=(8,4))
plt.hist(rp2, bins=100, alpha=0.5, density=True, label="Real")
plt.hist(fp2, bins=100, alpha=0.5, density=True, label="Generated")
plt.xlabel("Pixel intensity"); plt.ylabel("Density")
plt.title("Pixel intensity distribution"); plt.legend(); plt.tight_layout()
plt.savefig(OUTPUT_DIR/"pixel_distribution.png", dpi=120); plt.close()
print("saved pixel_distribution.png")